# Supplementary publication notebook
## Hilvan, Türkiye M5.3 — professional 2-D/3-D focal-mechanism and Coulomb-stress figures

This notebook is a **supplement to the executed focal-mechanism/Coulomb notebook**. It does not replace the source analysis. It reads the result tables and Coulomb grids already generated by that notebook and produces journal-scale graphics, tables, and diagrams.

### Current executed result carried by the parent analysis

The parent notebook reported:

- Event: `us6000tx9u`, 2026-09-24 07:40:58.138 UTC
- Epicenter: 37.4736°N, 38.8612°E
- Depth: 10 km
- Magnitude used as Mw proxy: 5.3
- Polarities used: **5**
- Final azimuth gap: **207.7°**
- Solution status: **EXPLORATORY_UNDERCONSTRAINED_AUTO**
- NP1: **118° / 66° / −50°**
- NP2: **233.86° / 45.59° / −145.29°**
- Near-best solutions: **1,588**
- Assumed stress drop: 3 MPa
- Effective friction: 0.4
- Estimated source dimensions: 6.36 × 3.18 km
- Estimated mean slip: 0.173 m

### Publication interpretation

The figures generated here are suitable as **exploratory/supplementary visualizations**, but the current focal mechanism is **not uniquely constrained**. Five automatic polarities and an azimuth gap greater than 180° allow many zero-misfit solutions. Before presenting the mechanism as a final source solution in an indexed geophysics paper, manually review the P polarities and ideally compare them with a regional moment-tensor solution.

This notebook intentionally carries that solution-status information into figure captions, metadata tables, and output filenames so that attractive graphics do not hide the uncertainty.

### Repository use
This notebook regenerates publication maps/tables from the archived derived results. It defaults to
`data/derived/study_snapshot/` and does not require raw MiniSEED/SAC files for the published figures.
Real web-tile basemaps require internet access; the analysis remains usable without them.

# 0. Packages

The notebook uses:

- NumPy, pandas, SciPy
- Matplotlib
- ObsPy
- PyProj
- Contextily + xyzservices for real web-tile basemaps
- Pyrocko for recomputing Coulomb stress at multiple depths
- Plotly for interactive 3-D visualization

The main static publication figures do **not** depend on Plotly.

In [ ]:
import sys, subprocess, importlib.util

REQUIRED = {
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "matplotlib": "matplotlib",
    "obspy": "obspy>=1.4.2",
    "pyproj": "pyproj",
    "contextily": "contextily",
    "xyzservices": "xyzservices",
    "plotly": "plotly",
    "tqdm": "tqdm",
}

missing = [pkg for mod, pkg in REQUIRED.items()
           if importlib.util.find_spec(mod) is None]

if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", *missing])
else:
    print("Core plotting packages are available.")

if importlib.util.find_spec("pyrocko") is None:
    print("\nPyrocko is required for the multi-depth Coulomb/3-D sections.")
    print("Trying pip install pyrocko ...")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "pyrocko"])
    except Exception as exc:
        print("Pyrocko installation failed:", exc)
        print("Conda alternative: conda install -c pyrocko pyrocko")

# 1. Locate the parent analysis results

Set `STUDY_ROOT` only if automatic discovery fails.

The required parent directory is the extracted event study containing:

`focal_coulomb_results/FINAL_FOCAL_COULOMB_SUMMARY.csv`

In [ ]:
from pathlib import Path
import json, math, warnings, os, re
import numpy as np
import pandas as pd

# -------- REPOSITORY DEFAULT --------
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

_default_study = REPO_ROOT / "data" / "derived" / "study_snapshot"
STUDY_ROOT = _default_study if (_default_study / "focal_coulomb_results").exists() else None
# Set STUDY_ROOT manually only if using another analysis directory.
# ------------------------------------

def looks_like_study(p):
    p = Path(p)
    return (
        p.is_dir()
        and (p / "focal_coulomb_results" / "FINAL_FOCAL_COULOMB_SUMMARY.csv").exists()
    )

def auto_find_study():
    roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    found = []
    for root in roots:
        try:
            for p in root.rglob("FINAL_FOCAL_COULOMB_SUMMARY.csv"):
                if p.parent.name == "focal_coulomb_results":
                    found.append(p.parent.parent)
        except Exception:
            pass
    # Prefer the Hilvan event.
    preferred = [p for p in found if "us6000tx9u" in str(p).lower()]
    return (preferred or found or [None])[0]

if STUDY_ROOT is None:
    STUDY_ROOT = auto_find_study()

if STUDY_ROOT is None:
    raise FileNotFoundError(
        "Could not find the parent focal_coulomb_results folder. "
        "Set STUDY_ROOT manually in this cell."
    )

STUDY_ROOT = Path(STUDY_ROOT)
RESULTS = STUDY_ROOT / "focal_coulomb_results"
TABLES_IN = RESULTS / "tables"
CFS_IN = RESULTS / "coulomb"

PUB = STUDY_ROOT / "publication_supplement"
FIG = PUB / "figures"
TAB = PUB / "tables"
DATA3D = PUB / "3d_data"
HTML = PUB / "interactive"

for p in [PUB, FIG, TAB, DATA3D, HTML]:
    p.mkdir(parents=True, exist_ok=True)

print("Study root:", STUDY_ROOT)
print("Publication output:", PUB)

# 2. Load and audit all parent results

In [ ]:
summary = pd.read_csv(RESULTS / "FINAL_FOCAL_COULOMB_SUMMARY.csv").iloc[0]

nodal = pd.read_csv(TABLES_IN / "nodal_planes.csv")
polarity = pd.read_csv(TABLES_IN / "best_mechanism_polarity_fit.csv")
near = pd.read_csv(TABLES_IN / "near_best_focal_solutions.csv")
site_best = pd.read_csv(TABLES_IN / "best_vertical_record_per_physical_site.csv")
cfs_parent = pd.read_csv(CFS_IN / "NP1_NP2_CFS_summary.csv")

EVENT_ID = str(summary["event_id"])
EVENT_LAT = float(summary["latitude"])
EVENT_LON = float(summary["longitude"])
EVENT_DEPTH_KM = float(summary["depth_km"])
EVENT_MAG = float(summary["event_magnitude"])
DOWNLOAD_RADIUS_KM = float(summary["download_radius_km"])

SOLUTION_STATUS = str(summary["focal_solution_status"])
N_POL = int(summary["n_polarities"])
AZ_GAP = float(summary["azimuth_gap_deg"])

NP1 = dict(
    strike=float(summary["NP1_strike"]),
    dip=float(summary["NP1_dip"]),
    rake=float(summary["NP1_rake"]),
)
NP2 = dict(
    strike=float(summary["NP2_strike"]),
    dip=float(summary["NP2_dip"]),
    rake=float(summary["NP2_rake"]),
)

STRESS_DROP_MPA = float(summary["assumed_stress_drop_MPa"])
EFFECTIVE_FRICTION = float(summary["effective_friction"])
SOURCE_LENGTH_KM = float(summary["source_length_km"])
SOURCE_WIDTH_KM = float(summary["source_width_km"])
SOURCE_SLIP_M = float(summary["source_mean_slip_m"])

publication_ready_mechanism = (
    N_POL >= 8
    and AZ_GAP < 180.0
    and "UNDERCONSTRAINED" not in SOLUTION_STATUS.upper()
    and "AUTO" not in SOLUTION_STATUS.upper()
)

print("Event:", EVENT_ID)
print("Mechanism status:", SOLUTION_STATUS)
print("Polarities:", N_POL)
print(f"Azimuth gap: {AZ_GAP:.1f}°")
print("Near-best solutions:", len(near))
print("Publication-ready focal mechanism:", publication_ready_mechanism)

if not publication_ready_mechanism:
    warnings.warn(
        "CURRENT MECHANISM IS EXPLORATORY/UNDERCONSTRAINED. "
        "Use these figures as exploratory/supplementary material until the "
        "first-motion dataset is manually reviewed and/or validated by a "
        "regional moment-tensor solution."
    )

# 3. Publication style and export helpers

Each static figure is exported as:

- 600-dpi PNG
- vector PDF
- vector SVG

Large fonts, thick lines, and readable legends are used for journal reduction.

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm, Normalize
from matplotlib.ticker import MaxNLocator, FuncFormatter
from matplotlib.lines import Line2D
from matplotlib.patches import Circle, Polygon, FancyArrowPatch
from matplotlib import patheffects

mpl.rcParams.update({
    "figure.figsize": (12, 9),
    "figure.dpi": 120,
    "savefig.dpi": 600,
    "font.size": 13,
    "axes.titlesize": 17,
    "axes.labelsize": 15,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 11,
    "axes.linewidth": 1.2,
    "lines.linewidth": 1.8,
    "lines.markersize": 8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

def save_figure(fig, stem, dpi=600):
    for ext in ("png", "pdf", "svg"):
        path = FIG / f"{stem}.{ext}"
        kw = dict(bbox_inches="tight")
        if ext == "png":
            kw["dpi"] = dpi
        fig.savefig(path, **kw)
    print("Saved:", stem, "(PNG/PDF/SVG)")

def panel_label(ax, label):
    ax.text(
        0.015, 0.985, label,
        transform=ax.transAxes,
        ha="left", va="top",
        fontsize=16, fontweight="bold",
        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="0.25", alpha=0.88),
        zorder=100,
    )

def status_box(ax, short=False):
    txt = (
        f"{SOLUTION_STATUS}\n"
        f"N={N_POL} polarities; azimuth gap={AZ_GAP:.1f}°"
    )
    if short:
        txt = f"{SOLUTION_STATUS} | N={N_POL} | gap={AZ_GAP:.1f}°"
    ax.text(
        0.99, 0.01, txt,
        transform=ax.transAxes,
        ha="right", va="bottom",
        fontsize=8.5,
        bbox=dict(boxstyle="round,pad=0.28", fc="white", ec="0.4", alpha=0.86),
        zorder=100,
    )

print("Figure output:", FIG)

# 4. Basemap helper

The real-basemap figures use web tiles through `contextily` when internet is available.

Default preference:
1. Esri World Hillshade / topographic background if available;
2. CartoDB Positron;
3. OpenStreetMap.

The provider attribution is retained by Contextily. For a submitted paper, verify the selected provider's license and required attribution in the final figure/caption.

In [ ]:
import contextily as cx
import xyzservices.providers as xyz
from pyproj import Transformer, CRS

WGS84 = CRS.from_epsg(4326)
WEB = CRS.from_epsg(3857)
to_web = Transformer.from_crs(WGS84, WEB, always_xy=True)
to_geo = Transformer.from_crs(WEB, WGS84, always_xy=True)

flat = xyz.flatten()

preferred_names = [
    "Esri.WorldHillshade",
    "Esri.WorldShadedRelief",
    "Esri.WorldTopoMap",
    "CartoDB.PositronNoLabels",
    "CartoDB.Positron",
    "OpenStreetMap.Mapnik",
]

BASEMAP_PROVIDER = None
BASEMAP_NAME = None
for name in preferred_names:
    if name in flat:
        BASEMAP_PROVIDER = flat[name]
        BASEMAP_NAME = name
        break

print("Selected basemap:", BASEMAP_NAME)
if BASEMAP_PROVIDER is not None:
    print("Attribution:", getattr(BASEMAP_PROVIDER, "attribution", "See provider metadata"))

def add_real_basemap(ax, zoom="auto", alpha=0.72):
    if BASEMAP_PROVIDER is None:
        return False
    try:
        cx.add_basemap(
            ax,
            source=BASEMAP_PROVIDER,
            crs=WEB,
            zoom=zoom,
            alpha=alpha,
            attribution_size=6,
        )
        return True
    except Exception as exc:
        print("Basemap unavailable; continuing without tiles:", exc)
        return False

## 4.1 Publication-controlled place names

Web-tile labels can become faint or disappear beneath the semi-transparent Coulomb-stress raster.  
The following gazetteer therefore redraws the most important regional places **above all map layers**.

Design rules:

- major regional cities use larger bold labels;
- local towns around the source use slightly smaller labels;
- every label has a white halo / semi-opaque white box for readability;
- labels are automatically filtered to the current map extent;
- 3-D figures use the same gazetteer projected into local east–north coordinates;
- the event epicenter remains a separate star symbol and is not replaced by a city marker.

Coordinates were curated from Wikidata/Wikipedia/OpenStreetMap-linked records and should be checked again before the final accepted manuscript if the journal requires an external gazetteer citation.

In [ ]:
# Curated regional gazetteer for reproducible publication labels.
# priority: 3 = major regional city, 2 = important nearby town, 1 = local town.
# dx/dy are annotation offsets in display points, chosen to reduce overlap.

PLACE_GAZETTEER = pd.DataFrame([
    {"name":"Hilvan",      "lat":37.58861, "lon":38.95556, "priority":2, "dx":  8, "dy":  9},
    {"name":"Şanlıurfa",   "lat":37.15833, "lon":38.79167, "priority":3, "dx":  9, "dy": -12},
    {"name":"Siverek",     "lat":37.75417, "lon":39.31778, "priority":2, "dx":  9, "dy":  8},
    {"name":"Bozova",      "lat":37.36250, "lon":38.52667, "priority":1, "dx": -42, "dy":  7},
    {"name":"Kâhta",       "lat":37.78028, "lon":38.62167, "priority":2, "dx": -36, "dy":  8},
    {"name":"Adıyaman",    "lat":37.76339, "lon":38.27714, "priority":3, "dx": -55, "dy":  8},
    {"name":"Diyarbakır",  "lat":37.91083, "lon":40.23667, "priority":3, "dx":  8, "dy":  8},
    {"name":"Gaziantep",   "lat":37.06278, "lon":37.37917, "priority":3, "dx": -58, "dy":  8},
    {"name":"Malatya",     "lat":38.35062, "lon":38.30940, "priority":3, "dx": -44, "dy":  8},
])

# Web Mercator coordinates used by all real-basemap maps.
_gx, _gy = to_web.transform(
    PLACE_GAZETTEER["lon"].to_numpy(float),
    PLACE_GAZETTEER["lat"].to_numpy(float),
)
PLACE_GAZETTEER["x3857"] = _gx
PLACE_GAZETTEER["y3857"] = _gy

# Local AEQD coordinates used by local 2-D/3-D stress figures.
LOCAL_AEQD = CRS.from_proj4(
    f"+proj=aeqd +lat_0={EVENT_LAT} +lon_0={EVENT_LON} "
    "+datum=WGS84 +units=m +no_defs"
)
to_local = Transformer.from_crs(WGS84, LOCAL_AEQD, always_xy=True)
_lx, _ly = to_local.transform(
    PLACE_GAZETTEER["lon"].to_numpy(float),
    PLACE_GAZETTEER["lat"].to_numpy(float),
)
PLACE_GAZETTEER["east_km"] = np.asarray(_lx)/1000.0
PLACE_GAZETTEER["north_km"] = np.asarray(_ly)/1000.0


def apply_lonlat_ticks(ax, decimals=2):
    """Display reader-friendly geographic longitude/latitude ticks on Web Mercator axes."""
    xmin, xmax = sorted(ax.get_xlim())
    ymin, ymax = sorted(ax.get_ylim())
    xmid = 0.5*(xmin+xmax)
    ymid = 0.5*(ymin+ymax)

    def _fmt_lon(x, pos=None):
        lon, _ = to_geo.transform(x, ymid)
        hemi = "E" if lon >= 0 else "W"
        return f"{abs(lon):.{decimals}f}°{hemi}"

    def _fmt_lat(y, pos=None):
        _, lat = to_geo.transform(xmid, y)
        hemi = "N" if lat >= 0 else "S"
        return f"{abs(lat):.{decimals}f}°{hemi}"

    ax.xaxis.set_major_formatter(FuncFormatter(_fmt_lon))
    ax.yaxis.set_major_formatter(FuncFormatter(_fmt_lat))
    ax.xaxis.set_major_locator(MaxNLocator(nbins=6))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=6))
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")

def places_in_web_extent(ax, min_priority=1):
    xmin, xmax = sorted(ax.get_xlim())
    ymin, ymax = sorted(ax.get_ylim())
    q = PLACE_GAZETTEER[
        (PLACE_GAZETTEER["priority"] >= min_priority)
        & (PLACE_GAZETTEER["x3857"] >= xmin)
        & (PLACE_GAZETTEER["x3857"] <= xmax)
        & (PLACE_GAZETTEER["y3857"] >= ymin)
        & (PLACE_GAZETTEER["y3857"] <= ymax)
    ].copy()
    return q.sort_values(["priority","name"], ascending=[False,True])

def annotate_places_web(
    ax,
    min_priority=1,
    marker=True,
    max_labels=None,
    fontsize_scale=1.0,
    zorder=40,
):
    # Draw clean publication place labels above basemap/stress overlays.
    q = places_in_web_extent(ax, min_priority=min_priority)
    if max_labels is not None:
        q = q.head(int(max_labels))

    artists = []
    for _, r in q.iterrows():
        if marker:
            size = 44 if int(r["priority"]) >= 3 else 30
            ax.scatter(
                [r["x3857"]], [r["y3857"]],
                s=size,
                marker="s" if int(r["priority"]) >= 3 else "o",
                facecolors="white",
                edgecolors="black",
                linewidths=1.1,
                zorder=zorder,
            )

        fs = (12.5 if int(r["priority"]) >= 3 else 11.0) * fontsize_scale
        fw = "bold" if int(r["priority"]) >= 2 else "semibold"

        ann = ax.annotate(
            r["name"],
            xy=(r["x3857"], r["y3857"]),
            xytext=(float(r["dx"]), float(r["dy"])),
            textcoords="offset points",
            ha="left",
            va="center",
            fontsize=fs,
            fontweight=fw,
            color="black",
            zorder=zorder+1,
            bbox=dict(
                boxstyle="round,pad=0.16",
                fc="white",
                ec="none",
                alpha=0.78,
            ),
            arrowprops=(
                dict(arrowstyle="-", color="0.20", lw=0.7, alpha=0.8)
                if abs(float(r["dx"])) >= 30 else None
            ),
        )
        ann.set_path_effects([
            patheffects.withStroke(linewidth=1.6, foreground="white")
        ])
        artists.append(ann)
    return artists

def places_in_local_extent(xlim, ylim, min_priority=1):
    xmin, xmax = sorted(xlim)
    ymin, ymax = sorted(ylim)
    q = PLACE_GAZETTEER[
        (PLACE_GAZETTEER["priority"] >= min_priority)
        & (PLACE_GAZETTEER["east_km"] >= xmin)
        & (PLACE_GAZETTEER["east_km"] <= xmax)
        & (PLACE_GAZETTEER["north_km"] >= ymin)
        & (PLACE_GAZETTEER["north_km"] <= ymax)
    ].copy()
    return q.sort_values(["priority","name"], ascending=[False,True])

def annotate_places_local_2d(
    ax,
    min_priority=1,
    fontsize_scale=0.82,
    zorder=30,
):
    # Place names on local East/North stress maps.
    q = places_in_local_extent(ax.get_xlim(), ax.get_ylim(), min_priority=min_priority)

    for _, r in q.iterrows():
        ax.scatter(
            r["east_km"], r["north_km"],
            s=26 if int(r["priority"]) >= 2 else 18,
            marker="s" if int(r["priority"]) >= 3 else "o",
            facecolors="white", edgecolors="black",
            linewidths=0.8, zorder=zorder,
        )
        fs = (10.5 if int(r["priority"]) >= 3 else 9.2) * fontsize_scale
        t = ax.annotate(
            r["name"],
            xy=(r["east_km"], r["north_km"]),
            xytext=(4,5),
            textcoords="offset points",
            fontsize=fs,
            fontweight="bold" if int(r["priority"]) >= 2 else "semibold",
            color="black",
            zorder=zorder+1,
            bbox=dict(boxstyle="round,pad=0.10", fc="white", ec="none", alpha=0.72),
        )
        t.set_path_effects([patheffects.withStroke(linewidth=1.2, foreground="white")])

def annotate_places_3d(
    ax,
    xlim=(-60,60),
    ylim=(-60,60),
    min_priority=1,
    z_surface=0.0,
    names=None,
):
    # Draw selected place names on the z=0 surface of a local 3-D plot.
    q = places_in_local_extent(xlim, ylim, min_priority=min_priority)
    if names is not None:
        q = q[q["name"].isin(list(names))]

    for _, r in q.iterrows():
        ax.scatter(
            [r["east_km"]], [r["north_km"]], [z_surface],
            s=34,
            c="white",
            edgecolors="black",
            linewidths=0.9,
            depthshade=False,
            zorder=50,
        )
        ax.text(
            r["east_km"], r["north_km"], z_surface+0.8,
            r["name"],
            fontsize=9.5 if int(r["priority"]) < 3 else 10.5,
            fontweight="bold",
            color="black",
            zorder=51,
        )

display(
    PLACE_GAZETTEER[
        ["name","lat","lon","priority","east_km","north_km"]
    ].sort_values(["priority","name"], ascending=[False,True])
)

# 5. Helper to load parent Coulomb grids

In [ ]:
def load_npz_for(plane_name):
    path = CFS_IN / f"{EVENT_ID}_{plane_name}_CFS.npz"
    if not path.exists():
        raise FileNotFoundError(path)
    return np.load(path, allow_pickle=True)

cfs_np1 = load_npz_for("NP1_waveform")
cfs_np2 = load_npz_for("NP2_auxiliary")

print("NP1 grid:", cfs_np1["cfs_mpa"].shape)
print("NP2 grid:", cfs_np2["cfs_mpa"].shape)

# Figure S1 — Regional station geometry and first-motion coverage

This figure combines:
- real topographic/hillshade basemap;
- downloaded-station geometry;
- polarities used in the focal solution;
- the 200-km acquisition circle;
- source location and focal mechanism inset.

Compression and dilatation are deliberately distinguished by **filled/open symbols**, not color alone.

In [ ]:
from obspy.imaging.beachball import beach

# Use all physical sites from parent QC table.
sites = site_best.copy()

# Event center in web mercator.
ex, ey = to_web.transform(EVENT_LON, EVENT_LAT)

# Convert station coordinates.
sx, sy = to_web.transform(
    sites["longitude"].to_numpy(float),
    sites["latitude"].to_numpy(float)
)
sites["x3857"] = sx
sites["y3857"] = sy

pol = polarity.copy()
px, py = to_web.transform(
    pol["longitude"].to_numpy(float),
    pol["latitude"].to_numpy(float)
)
pol["x3857"] = px
pol["y3857"] = py

# 200-km map extent approximately around event in projected metres.
pad = DOWNLOAD_RADIUS_KM * 1000 * 1.15

fig, ax = plt.subplots(figsize=(14, 12))

ax.set_xlim(ex-pad, ex+pad)
ax.set_ylim(ey-pad, ey+pad)

add_real_basemap(ax, alpha=0.75)

# Acquisition radius circle.
circle = Circle(
    (ex, ey), DOWNLOAD_RADIUS_KM*1000,
    fill=False, ec="0.15", lw=2.2, ls="--", zorder=5
)
ax.add_patch(circle)

# All physical sites.
ax.scatter(
    sites["x3857"], sites["y3857"],
    marker="^", s=75,
    facecolors="white", edgecolors="0.25",
    linewidths=1.1, alpha=0.9,
    label="Physical station sites", zorder=8
)

# Used polarities.
comp = pol[pol["polarity"] == 1]
dil = pol[pol["polarity"] == -1]

ax.scatter(
    comp["x3857"], comp["y3857"],
    marker="o", s=135,
    facecolors="black", edgecolors="white",
    linewidths=1.3, label="Compression (+)", zorder=12
)
ax.scatter(
    dil["x3857"], dil["y3857"],
    marker="o", s=135,
    facecolors="white", edgecolors="black",
    linewidths=2.0, label="Dilatation (−)", zorder=12
)

for _, r in pol.iterrows():
    t = ax.text(
        r["x3857"]+4500, r["y3857"]+4500,
        f'{r["station"]}',
        fontsize=10, fontweight="bold", zorder=15
    )
    t.set_path_effects([patheffects.withStroke(linewidth=3, foreground="white")])

ax.scatter(
    [ex], [ey], marker="*", s=420,
    facecolors="gold", edgecolors="black",
    linewidths=1.6, label="M5.3 mainshock", zorder=20
)

# Explicit place names remain readable above the basemap and stress layers.
annotate_places_web(
    ax,
    min_priority=1,
    max_labels=9,
    fontsize_scale=1.0,
    zorder=28,
)

# Focal beachball inset.
inset = ax.inset_axes([0.71, 0.69, 0.25, 0.25])
bb = beach(
    [NP1["strike"], NP1["dip"], NP1["rake"]],
    xy=(0,0), width=180, linewidth=1.2,
    facecolor="0.25"
)
inset.add_collection(bb)
inset.set_xlim(-105,105)
inset.set_ylim(-105,105)
inset.set_aspect("equal")
inset.axis("off")
inset.set_title(
    f'NP1 {NP1["strike"]:.0f}/{NP1["dip"]:.0f}/{NP1["rake"]:.0f}',
    fontsize=11, fontweight="bold"
)

apply_lonlat_ticks(ax, decimals=2)
ax.set_title(
    "Regional station geometry and first-motion observations\n"
    "Hilvan M5.3, 24 September 2026",
    fontweight="bold"
)
ax.legend(loc="lower left", frameon=True, framealpha=0.95, ncol=2)
status_box(ax)
panel_label(ax, "S1")

save_figure(fig, "FigS01_Regional_station_geometry_real_basemap")
plt.show()

# Figure S2 — Focal-sphere azimuth/takeoff coverage and azimuth gap

The left panel is a lower-hemisphere style polar coverage plot.  
The right panel explicitly shows the azimuth sampling gap.

This diagnostic figure is essential because the mechanism currently has an azimuth gap greater than 180°.

In [ ]:
fig = plt.figure(figsize=(15, 7))
ax1 = fig.add_subplot(1,2,1, projection="polar")
ax2 = fig.add_subplot(1,2,2, projection="polar")

# Lower-hemisphere style: radius = takeoff mapped toward center.
theta = np.deg2rad(pol["azimuth_deg"].to_numpy(float))
r = 90.0 - np.clip(pol["takeoff_angle_deg"].to_numpy(float), 0, 180)

for _, rr in pol.iterrows():
    th = np.deg2rad(rr["azimuth_deg"])
    rad = 90.0 - np.clip(rr["takeoff_angle_deg"], 0, 180)
    if rr["polarity"] == 1:
        ax1.scatter(th, rad, s=160, c="black", edgecolors="white", linewidths=1.2, zorder=5)
    else:
        ax1.scatter(th, rad, s=160, facecolors="white", edgecolors="black", linewidths=2.0, zorder=5)
    ax1.text(th, rad+5, rr["station"], ha="center", va="center", fontsize=10, fontweight="bold")

ax1.set_theta_zero_location("N")
ax1.set_theta_direction(-1)
ax1.set_ylim(-90, 90)
ax1.set_yticks([-60,-30,0,30,60])
ax1.set_yticklabels(["150°","120°","90°","60°","30°"])
ax1.set_title("P-polarity focal-sphere sampling", pad=22, fontweight="bold")
panel_label(ax1, "a")

# Azimuth gap panel.
az = np.sort(np.mod(pol["azimuth_deg"].to_numpy(float),360))
gaps = np.diff(np.r_[az, az[0]+360])
imax = int(np.argmax(gaps))
gap_start = az[imax]
gap_end = (az[(imax+1)%len(az)]) % 360
gap_size = gaps[imax]

ax2.scatter(theta, np.ones_like(theta), s=150, c="black", zorder=5)
for _, rr in pol.iterrows():
    ax2.text(np.deg2rad(rr["azimuth_deg"]), 1.08, rr["station"],
             ha="center", va="center", fontsize=10, fontweight="bold")

# Shade missing sector.
start = np.deg2rad(gap_start)
width = np.deg2rad(gap_size)
ax2.bar(
    start, 1.0, width=width, bottom=0,
    align="edge", alpha=0.25, edgecolor="0.2",
    linewidth=1.5, zorder=1
)

ax2.set_theta_zero_location("N")
ax2.set_theta_direction(-1)
ax2.set_ylim(0,1.25)
ax2.set_yticks([])
ax2.set_title(
    f"Azimuth coverage\nmaximum gap = {gap_size:.1f}°",
    pad=22, fontweight="bold"
)
panel_label(ax2, "b")

fig.suptitle(
    "Focal-mechanism observational geometry",
    fontsize=19, fontweight="bold", y=1.03
)
fig.text(
    0.5, -0.01,
    f"{SOLUTION_STATUS} — mechanism geometry remains underconstrained",
    ha="center", fontsize=11
)
fig.tight_layout()

save_figure(fig, "FigS02_Focal_sphere_and_azimuth_gap")
plt.show()

# Figure S3 — Focal-mechanism non-uniqueness

The parent inversion found many equally fitting solutions.  
This figure visualizes the near-best strike–dip–rake family instead of showing only one beachball.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

h1 = axes[0].hist2d(
    near["strike"], near["dip"],
    bins=[36,18], cmap="viridis"
)
axes[0].scatter(NP1["strike"], NP1["dip"], marker="*", s=220, c="red", edgecolors="black")
axes[0].set_xlabel("Strike (°)")
axes[0].set_ylabel("Dip (°)")
axes[0].set_title("Near-best strike–dip density", fontweight="bold")
fig.colorbar(h1[3], ax=axes[0], label="Solution count")
panel_label(axes[0], "a")

h2 = axes[1].hist2d(
    near["strike"], near["rake"],
    bins=[36,36], cmap="viridis"
)
axes[1].scatter(NP1["strike"], NP1["rake"], marker="*", s=220, c="red", edgecolors="black")
axes[1].set_xlabel("Strike (°)")
axes[1].set_ylabel("Rake (°)")
axes[1].set_title("Near-best strike–rake density", fontweight="bold")
fig.colorbar(h2[3], ax=axes[1], label="Solution count")
panel_label(axes[1], "b")

h3 = axes[2].hist2d(
    near["dip"], near["rake"],
    bins=[18,36], cmap="viridis"
)
axes[2].scatter(NP1["dip"], NP1["rake"], marker="*", s=220, c="red", edgecolors="black")
axes[2].set_xlabel("Dip (°)")
axes[2].set_ylabel("Rake (°)")
axes[2].set_title("Near-best dip–rake density", fontweight="bold")
fig.colorbar(h3[3], ax=axes[2], label="Solution count")
panel_label(axes[2], "c")

fig.suptitle(
    f"Focal-mechanism non-uniqueness: {len(near):,} near-best solutions",
    fontsize=19, fontweight="bold"
)
fig.tight_layout()

save_figure(fig, "FigS03_Focal_mechanism_solution_density")
plt.show()

# 6. Real-basemap Coulomb figure helper

The parent ΔCFS grids are projected to Web Mercator and drawn over a real hillshade/topographic basemap.

For the paper:
- use a symmetric diverging scale centered on 0;
- show contours at ±0.01 MPa when they exist;
- retain a second zoomed scale if needed because Coulomb stress is strongly concentrated near the source.

In [ ]:
def projected_cfs_grid(npz):
    lon = npz["longitude"]
    lat = npz["latitude"]
    cfs = npz["cfs_mpa"]
    x, y = to_web.transform(lon, lat)
    return np.asarray(x), np.asarray(y), np.asarray(cfs)

def robust_cfs_limit(cfs, percentile=99.0):
    z = np.abs(cfs[np.isfinite(cfs)])
    if len(z) == 0:
        return 1.0
    v = np.nanpercentile(z, percentile)
    if not np.isfinite(v) or v <= 0:
        v = np.nanmax(z)
    return float(v)

def plot_cfs_real_basemap(npz, plane, name, figure_label):
    X, Y, C = projected_cfs_grid(npz)
    ex, ey = to_web.transform(EVENT_LON, EVENT_LAT)

    vmax = robust_cfs_limit(C, 99.0)

    fig, ax = plt.subplots(figsize=(14, 12))
    ax.set_xlim(np.nanmin(X), np.nanmax(X))
    ax.set_ylim(np.nanmin(Y), np.nanmax(Y))

    add_real_basemap(ax, alpha=0.72)

    im = ax.pcolormesh(
        X, Y, C,
        cmap="RdBu_r",
        norm=TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax),
        shading="auto",
        alpha=0.78,
        zorder=6,
        rasterized=True,
    )

    # zero contour and threshold contours.
    try:
        ax.contour(X, Y, C, levels=[0], colors="black", linewidths=1.0, zorder=9)
        levels = [v for v in (-0.01, 0.01) if np.nanmin(C) < v < np.nanmax(C)]
        if levels:
            cs = ax.contour(X, Y, C, levels=levels, colors="black",
                            linewidths=1.2, linestyles="--", zorder=10)
            ax.clabel(cs, fmt=lambda v: f"{v:+.2f} MPa", fontsize=9)
    except Exception:
        pass

    ax.scatter(
        [ex], [ey],
        marker="*", s=400,
        facecolors="gold", edgecolors="black",
        linewidths=1.5, zorder=20,
        label="M5.3 hypocentral projection"
    )

    # Strike line at source.
    ph = np.deg2rad(plane["strike"])
    halfL = SOURCE_LENGTH_KM*500.0
    nn = np.array([-halfL*np.cos(ph), halfL*np.cos(ph)])
    ee = np.array([-halfL*np.sin(ph), halfL*np.sin(ph)])

    # local AEQD for the source line.
    local = CRS.from_proj4(
        f"+proj=aeqd +lat_0={EVENT_LAT} +lon_0={EVENT_LON} +datum=WGS84 +units=m +no_defs"
    )
    local_to_geo = Transformer.from_crs(local, WGS84, always_xy=True)
    flon, flat = local_to_geo.transform(ee, nn)
    fx, fy = to_web.transform(flon, flat)
    ax.plot(fx, fy, color="black", lw=4.0, zorder=18, label="Source strike")

    # Draw city/town names last so the ΔCFS raster cannot hide them.
    annotate_places_web(
        ax,
        min_priority=1,
        max_labels=9,
        fontsize_scale=1.0,
        zorder=30,
    )

    cb = fig.colorbar(im, ax=ax, shrink=0.82, pad=0.02)
    cb.set_label("Coulomb failure stress change, ΔCFS (MPa)", fontsize=14)

    apply_lonlat_ticks(ax, decimals=2)
    ax.set_title(
        f"{name}: Coulomb failure stress on real basemap\n"
        f'Strike/Dip/Rake = {plane["strike"]:.1f}° / {plane["dip"]:.1f}° / {plane["rake"]:.1f}°',
        fontweight="bold"
    )
    ax.legend(loc="upper right", framealpha=0.95)
    panel_label(ax, figure_label)
    status_box(ax)

    return fig, ax

fig, ax = plot_cfs_real_basemap(
    cfs_np1, NP1, "NP1 waveform scenario", "S4"
)
save_figure(fig, "FigS04_Coulomb_NP1_real_basemap")
plt.show()

fig, ax = plot_cfs_real_basemap(
    cfs_np2, NP2, "NP2 auxiliary-plane scenario", "S5"
)
save_figure(fig, "FigS05_Coulomb_NP2_real_basemap")
plt.show()

# Figure S6 — NP1 versus NP2 comparison on the same quantitative scale

This comparison uses one symmetric color scale for both planes so the lobe patterns can be compared without visual rescaling.

In [ ]:
X1, Y1, C1 = projected_cfs_grid(cfs_np1)
X2, Y2, C2 = projected_cfs_grid(cfs_np2)
ex, ey = to_web.transform(EVENT_LON, EVENT_LAT)

joint_v = np.nanpercentile(
    np.abs(np.r_[C1[np.isfinite(C1)].ravel(), C2[np.isfinite(C2)].ravel()]),
    99
)

fig, axes = plt.subplots(1,2, figsize=(19, 9), sharex=True, sharey=True)

for ax, X, Y, C, plane, title, lab in [
    (axes[0], X1,Y1,C1,NP1,"NP1 waveform scenario","a"),
    (axes[1], X2,Y2,C2,NP2,"NP2 auxiliary scenario","b"),
]:
    ax.set_xlim(np.nanmin(X1),np.nanmax(X1))
    ax.set_ylim(np.nanmin(Y1),np.nanmax(Y1))
    add_real_basemap(ax, alpha=0.68)
    im = ax.pcolormesh(
        X,Y,C, shading="auto",
        cmap="RdBu_r",
        norm=TwoSlopeNorm(vmin=-joint_v,vcenter=0,vmax=joint_v),
        alpha=0.78, rasterized=True
    )
    try:
        ax.contour(X,Y,C,levels=[0],colors="black",linewidths=0.9)
    except Exception:
        pass
    ax.scatter(ex,ey,marker="*",s=300,facecolors="gold",edgecolors="black",zorder=15)

    annotate_places_web(
        ax,
        min_priority=1,
        max_labels=8,
        fontsize_scale=0.80,
        zorder=25,
    )

    ax.set_title(
        f'{title}\n{plane["strike"]:.1f}/{plane["dip"]:.1f}/{plane["rake"]:.1f}',
        fontweight="bold"
    )
    panel_label(ax, lab)
    apply_lonlat_ticks(ax, decimals=2)

# Longitude/latitude tick formatting has already been applied panel-by-panel.
cb = fig.colorbar(im, ax=axes, shrink=0.82, pad=0.02)
cb.set_label("ΔCFS (MPa)")
fig.suptitle(
    "Alternative nodal-plane Coulomb-stress scenarios",
    fontsize=19, fontweight="bold"
)
fig.text(
    0.5, 0.02,
    f"{SOLUTION_STATUS}; the physical rupture plane is not independently resolved.",
    ha="center", fontsize=10
)
fig.subplots_adjust(top=0.88, bottom=0.10, wspace=0.06)

save_figure(fig, "FigS06_NP1_NP2_Coulomb_comparison_real_basemap")
plt.show()

# 7. Recompute Coulomb stress at multiple depths for 3-D visualization

The parent analysis stored a 10-km receiver-depth plane.  
For a genuine 3-D visualization, this supplement recomputes ΔCFS at multiple receiver depths using the same source assumptions.

**This is a model-volume visualization, not a 3-D inversion of stress.**

In [ ]:
try:
    from pyrocko.modelling import OkadaSource, okada_ext
    PYROCKO_OK = True
except Exception as exc:
    PYROCKO_OK = False
    print("Pyrocko unavailable:", exc)

SHEAR_MODULUS_PA = 32e9
POISSON = 0.25
ASPECT_RATIO = 2.0
PORE_PRESSURE_CHANGE_PA = 0.0

def mw_to_m0(mw):
    return 10.0**(1.5*mw + 9.1)

def estimate_source_scale(mw, stress_drop_mpa):
    m0 = mw_to_m0(mw)
    ds = stress_drop_mpa*1e6
    r = (7.0*m0/(16.0*ds))**(1.0/3.0)
    area = np.pi*r*r
    L = np.sqrt(area*ASPECT_RATIO)
    W = area/L
    slip = m0/(SHEAR_MODULUS_PA*area)
    return L, W, slip

def lame_lambda(mu, nu):
    return 2.0*mu*nu/(1.0-2.0*nu)

def receiver_vectors(strike,dip,rake):
    ph, de, ra = np.deg2rad([strike,dip,rake])

    ns = np.array([
        np.sin(de)*np.cos(ph+0.5*np.pi),
        np.sin(de)*np.sin(ph+0.5*np.pi),
        -np.cos(de)
    ])

    rst = np.array([np.cos(ph),np.sin(ph),0.0])
    rdi = np.array([
        np.cos(de)*np.cos(ph+0.5*np.pi),
        np.cos(de)*np.sin(ph+0.5*np.pi),
        np.sin(de)
    ])
    ts = rst*np.cos(ra)-rdi*np.sin(ra)
    return ns,ts

def cfs_depth_plane(plane, depth_km, half_width_km=60, grid_n=121):
    if not PYROCKO_OK:
        raise RuntimeError("Pyrocko is required for multi-depth Coulomb calculation.")

    L,W,D = estimate_source_scale(EVENT_MAG,STRESS_DROP_MPA)

    source = OkadaSource(
        lat=EVENT_LAT, lon=EVENT_LON,
        north_shift=0.0, east_shift=0.0,
        depth=EVENT_DEPTH_KM*1000.0,
        al1=-L/2, al2=L/2,
        aw1=-W/2, aw2=W/2,
        strike=plane["strike"], dip=plane["dip"], rake=plane["rake"],
        slip=D, opening=0.0,
        poisson=POISSON, shearmod=SHEAR_MODULUS_PA
    )

    axis = np.linspace(-half_width_km,half_width_km,grid_n)*1000
    EE,NN = np.meshgrid(axis,axis)
    rec = np.column_stack([
        NN.ravel(), EE.ravel(),
        np.full(EE.size, depth_km*1000.0)
    ])

    lam = lame_lambda(SHEAR_MODULUS_PA,POISSON)

    result = okada_ext.okada(
        source.source_patch()[None,:],
        source.source_disloc()[None,:],
        rec, lam, SHEAR_MODULUS_PA,
        nthreads=0, rotate_sdn=False, stack_sources=True
    )

    grad = result[:,3:12]
    gradT = result[:,(3,6,9,4,7,10,5,8,11)]
    strain = 0.5*(grad+gradT)

    diag=[0,4,8]
    dil = strain[:,diag].sum(axis=1)[:,None]
    I=np.zeros(9); I[diag]=1.0
    stress = I[None,:]*lam*dil + 2*SHEAR_MODULUS_PA*strain

    ns,ts = receiver_vectors(plane["strike"],plane["dip"],plane["rake"])
    sigma_n=np.sum(np.tile(ns,3)*stress*np.repeat(ns,3),axis=1)
    tau=np.sum(np.tile(ts,3)*stress*np.repeat(ns,3),axis=1)

    cfs = tau + EFFECTIVE_FRICTION*(sigma_n + PORE_PRESSURE_CHANGE_PA)

    return EE/1000.0, NN/1000.0, cfs.reshape(grid_n,grid_n)/1e6

DEPTHS_KM = [2.0, 5.0, 10.0, 15.0, 20.0]
print("Depth slices:", DEPTHS_KM)

# Figure S7 — Multi-depth Coulomb slices

A large-format five-panel figure shows how the modelled stress field changes with receiver depth.

In [ ]:
depth_fields = {}

if PYROCKO_OK:
    for dep in DEPTHS_KM:
        E,N,C = cfs_depth_plane(NP1,dep)
        depth_fields[dep]=(E,N,C)

    common = np.nanpercentile(
        np.abs(np.concatenate([v[2].ravel() for v in depth_fields.values()])),
        99
    )

    fig, axes = plt.subplots(1,5, figsize=(25,5.5), sharex=True, sharey=True)

    for ax, dep in zip(axes,DEPTHS_KM):
        E,N,C=depth_fields[dep]
        im=ax.pcolormesh(
            E,N,C,shading="auto",cmap="RdBu_r",
            norm=TwoSlopeNorm(vmin=-common,vcenter=0,vmax=common),
            rasterized=True
        )
        ax.contour(E,N,C,levels=[0],colors="black",linewidths=0.8)
        ax.scatter(0,0,marker="*",s=150,c="gold",edgecolors="black",zorder=25)

        annotate_places_local_2d(
            ax,
            min_priority=1,
            fontsize_scale=0.66,
            zorder=27,
        )

        ax.set_title(f"{dep:g} km depth",fontweight="bold")
        ax.set_xlabel("East (km)")
        ax.set_aspect("equal")
        ax.grid(alpha=0.15)

    axes[0].set_ylabel("North (km)")
    cb=fig.colorbar(im,ax=axes,shrink=0.8,pad=0.015)
    cb.set_label("ΔCFS (MPa)")
    fig.suptitle(
        "NP1 Coulomb-stress evolution with receiver depth",
        fontsize=19,fontweight="bold"
    )
    fig.subplots_adjust(top=0.83,wspace=0.08)

    save_figure(fig,"FigS07_Coulomb_multi_depth_slices")
    plt.show()
else:
    print("Skipped: Pyrocko not available.")

# Figure S8 — Advanced static 3-D Coulomb stack

The five depth planes are drawn as horizontal surfaces in a true 3-D figure.  
Depth increases downward.

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from matplotlib.cm import ScalarMappable

if depth_fields:
    common = np.nanpercentile(
        np.abs(np.concatenate([v[2].ravel() for v in depth_fields.values()])),
        99
    )
    norm=TwoSlopeNorm(vmin=-common,vcenter=0,vmax=common)
    cmap=plt.get_cmap("RdBu_r")

    fig=plt.figure(figsize=(14,11))
    ax=fig.add_subplot(111,projection="3d")

    # Subsample for performant vector rendering.
    step=4

    for dep in DEPTHS_KM:
        E,N,C=depth_fields[dep]
        ax.plot_surface(
            E[::step,::step],
            N[::step,::step],
            np.full_like(E[::step,::step],-dep),
            facecolors=cmap(norm(C[::step,::step])),
            rstride=1,cstride=1,
            linewidth=0,antialiased=False,
            shade=False,alpha=0.78
        )

    # Source/hypocenter.
    ax.scatter(
        [0],[0],[-EVENT_DEPTH_KM],
        marker="*",s=220,c="gold",edgecolors="black",
        depthshade=False,label="Mainshock"
    )

    annotate_places_3d(
        ax,
        xlim=(-60,60),
        ylim=(-60,60),
        min_priority=1,
        z_surface=0.0,
        names=["Hilvan","Şanlıurfa","Siverek","Bozova","Kâhta","Adıyaman"],
    )

    ax.set_xlabel("East (km)",labelpad=12)
    ax.set_ylabel("North (km)",labelpad=12)
    ax.set_zlabel("Elevation relative to surface (km)",labelpad=12)
    ax.set_title(
        "3-D stacked Coulomb-stress field — NP1 scenario",
        fontweight="bold",pad=18
    )
    ax.view_init(elev=26,azim=-55)

    sm=ScalarMappable(norm=norm,cmap=cmap); sm.set_array([])
    cb=fig.colorbar(sm,ax=ax,shrink=0.65,pad=0.08)
    cb.set_label("ΔCFS (MPa)")

    ax.legend(loc="upper left")
    fig.tight_layout()

    save_figure(fig,"FigS08_3D_Coulomb_depth_stack")
    plt.show()

# Figure S9 — 3-D Coulomb iso-cloud + source plane

For an intuitive structural view, points with comparatively strong positive and negative ΔCFS are plotted in 3-D, together with the finite rectangular source plane.

This is a visualization of the calculated elastic model, not an inversion volume.

In [ ]:
def source_plane_xyz(plane, length_km, width_km, center_depth_km):
    strike=np.deg2rad(plane["strike"])
    dip=np.deg2rad(plane["dip"])

    # axes in ENDown coordinates
    along=np.array([np.sin(strike),np.cos(strike),0.0])
    downdip=np.array([
        np.cos(strike)*np.cos(dip),
        -np.sin(strike)*np.cos(dip),
        np.sin(dip)
    ])

    corners=[]
    for a in (-0.5,0.5):
        for w in (-0.5,0.5):
            p=a*length_km*along + w*width_km*downdip
            corners.append(p)

    # reorder as rectangle perimeter
    order=[0,1,3,2]
    q=np.array([corners[i] for i in order])
    E=q[:,0]
    N=q[:,1]
    Z=-(center_depth_km+q[:,2])
    return E,N,Z

if depth_fields:
    pts=[]
    for dep,(E,N,C) in depth_fields.items():
        # keep only strong tails to prevent an unreadable cloud
        thr=np.nanpercentile(np.abs(C),97.5)
        mask=np.abs(C)>=thr
        for e,n,c in zip(E[mask],N[mask],C[mask]):
            pts.append((e,n,-dep,c))

    pts=np.array(pts,float)

    fig=plt.figure(figsize=(14,11))
    ax=fig.add_subplot(111,projection="3d")

    lim=np.nanpercentile(np.abs(pts[:,3]),98)
    norm=TwoSlopeNorm(vmin=-lim,vcenter=0,vmax=lim)

    sc=ax.scatter(
        pts[:,0],pts[:,1],pts[:,2],
        c=pts[:,3],cmap="RdBu_r",norm=norm,
        s=16,alpha=0.55,depthshade=False
    )

    e,n,z=source_plane_xyz(
        NP1,SOURCE_LENGTH_KM,SOURCE_WIDTH_KM,EVENT_DEPTH_KM
    )
    ax.add_collection3d(
        mpl.collections.PolyCollection(
            [np.column_stack([e,n])],
            facecolors="none",edgecolors="black",linewidths=2.5
        ),
        zs=np.mean(z),zdir="z"
    )
    # also draw exact 3D perimeter
    ax.plot(np.r_[e,e[0]],np.r_[n,n[0]],np.r_[z,z[0]],
            c="black",lw=3,label="Approximate source plane")

    ax.scatter([0],[0],[-EVENT_DEPTH_KM],
               marker="*",s=250,c="gold",edgecolors="black",depthshade=False)

    annotate_places_3d(
        ax,
        xlim=(-60,60),
        ylim=(-60,60),
        min_priority=1,
        z_surface=0.0,
        names=["Hilvan","Şanlıurfa","Siverek","Bozova","Kâhta","Adıyaman"],
    )

    ax.set_xlabel("East (km)")
    ax.set_ylabel("North (km)")
    ax.set_zlabel("Depth sign convention: surface = 0 km")
    ax.set_title(
        "3-D high-amplitude Coulomb-stress lobes and source geometry",
        fontweight="bold",pad=18
    )
    ax.view_init(elev=23,azim=-48)
    ax.legend()

    cb=fig.colorbar(sc,ax=ax,shrink=0.65,pad=0.08)
    cb.set_label("ΔCFS (MPa)")
    fig.tight_layout()

    save_figure(fig,"FigS09_3D_Coulomb_lobes_source_plane")
    plt.show()

# Figure S10 — Along-strike and across-strike Coulomb cross-sections

Cross-sections are often easier to interpret scientifically than perspective 3-D plots.

The supplement interpolates the multi-depth NP1 volume along:
- the NP1 strike direction;
- a perpendicular profile.

In [ ]:
from scipy.interpolate import RegularGridInterpolator

if depth_fields:
    depths=np.array(DEPTHS_KM,float)

    # all depth fields use same E,N grid
    E,N,_=depth_fields[DEPTHS_KM[0]]
    eaxis=E[0,:]
    naxis=N[:,0]

    volume=np.stack([depth_fields[d][2] for d in DEPTHS_KM],axis=0)
    # interpolator axes = depth, north, east
    interp=RegularGridInterpolator(
        (depths,naxis,eaxis),
        volume,bounds_error=False,fill_value=np.nan
    )

    dist=np.linspace(-55,55,321)
    zz=np.linspace(min(depths),max(depths),161)
    DD,ZZ=np.meshgrid(dist,zz)

    strike=np.deg2rad(NP1["strike"])
    # positive profile direction along strike
    Ealong=DD*np.sin(strike)
    Nalong=DD*np.cos(strike)

    # cross strike
    Ecross=DD*np.cos(strike)
    Ncross=-DD*np.sin(strike)

    pts_along=np.column_stack([ZZ.ravel(),Nalong.ravel(),Ealong.ravel()])
    pts_cross=np.column_stack([ZZ.ravel(),Ncross.ravel(),Ecross.ravel()])

    Calong=interp(pts_along).reshape(ZZ.shape)
    Ccross=interp(pts_cross).reshape(ZZ.shape)

    vmax=np.nanpercentile(np.abs(np.r_[Calong.ravel(),Ccross.ravel()]),99)

    fig,axes=plt.subplots(1,2,figsize=(18,7),sharey=True)

    for ax,C,title,lab in [
        (axes[0],Calong,"Along NP1 strike","a"),
        (axes[1],Ccross,"Across NP1 strike","b"),
    ]:
        im=ax.pcolormesh(
            DD,ZZ,C,shading="auto",cmap="RdBu_r",
            norm=TwoSlopeNorm(vmin=-vmax,vcenter=0,vmax=vmax)
        )
        ax.contour(DD,ZZ,C,levels=[0],colors="black",linewidths=0.9)
        ax.scatter([0],[EVENT_DEPTH_KM],marker="*",s=180,c="gold",edgecolors="black")
        ax.invert_yaxis()
        ax.set_xlabel("Profile distance (km)")
        ax.set_title(title,fontweight="bold")
        panel_label(ax,lab)

    axes[0].set_ylabel("Depth (km)")
    cb=fig.colorbar(im,ax=axes,shrink=0.82,pad=0.02)
    cb.set_label("ΔCFS (MPa)")
    fig.suptitle(
        "Vertical Coulomb-stress cross-sections — NP1 scenario",
        fontsize=19,fontweight="bold"
    )
    fig.subplots_adjust(top=0.86,wspace=0.08)

    save_figure(fig,"FigS10_Coulomb_vertical_cross_sections")
    plt.show()

# Figure S11 — Interactive 3-D Coulomb volume

This HTML figure is useful for author inspection and online supplements.  
Most journals still require the static 3-D figure for the manuscript PDF.

In [ ]:
if depth_fields:
    import plotly.graph_objects as go

    # Build sparse cloud from all slices.
    xs=[]; ys=[]; zs=[]; vals=[]
    for dep,(E,N,C) in depth_fields.items():
        step=3
        xs.append(E[::step,::step].ravel())
        ys.append(N[::step,::step].ravel())
        zs.append(np.full(E[::step,::step].size,-dep))
        vals.append(C[::step,::step].ravel())

    x=np.concatenate(xs)
    y=np.concatenate(ys)
    z=np.concatenate(zs)
    v=np.concatenate(vals)

    cutoff=np.nanpercentile(np.abs(v),92)
    keep=np.abs(v)>=cutoff

    fig3d=go.Figure()

    fig3d.add_trace(go.Scatter3d(
        x=x[keep],y=y[keep],z=z[keep],
        mode="markers",
        marker=dict(
            size=3,
            color=v[keep],
            colorscale="RdBu",
            cmin=-np.nanpercentile(np.abs(v),99),
            cmax=np.nanpercentile(np.abs(v),99),
            colorbar=dict(title="ΔCFS (MPa)"),
            opacity=0.55
        ),
        name="Strong ΔCFS"
    ))

    e,n,zp=source_plane_xyz(
        NP1,SOURCE_LENGTH_KM,SOURCE_WIDTH_KM,EVENT_DEPTH_KM
    )
    fig3d.add_trace(go.Scatter3d(
        x=np.r_[e,e[0]],y=np.r_[n,n[0]],z=np.r_[zp,zp[0]],
        mode="lines",
        line=dict(color="black",width=7),
        name="Source plane"
    ))

    fig3d.add_trace(go.Scatter3d(
        x=[0],y=[0],z=[-EVENT_DEPTH_KM],
        mode="markers",
        marker=dict(size=8,color="gold",symbol="diamond"),
        name="Mainshock"
    ))

    # Add place markers and names to the z=0 surface.
    p3 = places_in_local_extent((-60,60),(-60,60),min_priority=1)
    p3 = p3[p3["name"].isin(["Hilvan","Şanlıurfa","Siverek","Bozova","Kâhta","Adıyaman"])]
    if not p3.empty:
        fig3d.add_trace(go.Scatter3d(
            x=p3["east_km"],
            y=p3["north_km"],
            z=np.zeros(len(p3)),
            mode="markers+text",
            text=p3["name"],
            textposition="top center",
            textfont=dict(size=13,color="black"),
            marker=dict(
                size=5,
                color="white",
                line=dict(color="black",width=1.2)
            ),
            name="Places",
            hovertemplate="%{text}<extra></extra>",
        ))

    fig3d.update_layout(
        title="Hilvan M5.3 — interactive NP1 Coulomb-stress volume",
        width=1100,height=850,
        scene=dict(
            xaxis_title="East (km)",
            yaxis_title="North (km)",
            zaxis_title="Elevation relative to surface (km)",
            aspectmode="data"
        )
    )

    html_path=HTML/"FigS11_interactive_3D_Coulomb.html"
    fig3d.write_html(html_path,include_plotlyjs="cdn")
    print("Saved:",html_path)
    fig3d.show()

# Figure S12 — Analysis workflow diagram

A manuscript-ready vector workflow from waveform data to Coulomb-stress interpretation.

In [ ]:
from matplotlib.patches import FancyBboxPatch

fig,ax=plt.subplots(figsize=(18,6))
ax.set_xlim(0,18)
ax.set_ylim(0,6)
ax.axis("off")

steps=[
    (0.5,2.0,2.4,2.0,"Waveforms\nMiniSEED + StationXML"),
    (3.4,2.0,2.4,2.0,"P arrival\nresponse correction"),
    (6.3,2.0,2.4,2.0,"First motion\n+ / − polarity"),
    (9.2,2.0,2.4,2.0,"Double-couple\nSDR inversion"),
    (12.1,2.0,2.4,2.0,"NP1 + NP2\nuncertainty"),
    (15.0,2.0,2.4,2.0,"Okada ΔCFS\n2-D + 3-D"),
]

for x,y,w,h,label in steps:
    patch=FancyBboxPatch(
        (x,y),w,h,
        boxstyle="round,pad=0.15,rounding_size=0.15",
        fc="white",ec="black",lw=2
    )
    ax.add_patch(patch)
    ax.text(x+w/2,y+h/2,label,ha="center",va="center",
            fontsize=13,fontweight="bold")

for i in range(len(steps)-1):
    x1=steps[i][0]+steps[i][2]
    x2=steps[i+1][0]
    y=3.0
    ax.add_patch(FancyArrowPatch(
        (x1+0.08,y),(x2-0.08,y),
        arrowstyle="-|>",mutation_scale=20,lw=2
    ))

ax.text(
    9,5.25,
    "Hilvan M5.3 source-to-stress analysis workflow",
    ha="center",fontsize=20,fontweight="bold"
)
ax.text(
    9,0.65,
    "Current focal solution is exploratory: manual polarity review / moment-tensor validation required before final interpretation.",
    ha="center",fontsize=11
)

save_figure(fig,"FigS12_Analysis_workflow_diagram")
plt.show()

# 8. Publication tables

The following tables are exported as:
- CSV
- LaTeX (`.tex`)
- high-resolution PNG table figure where appropriate

In [ ]:
def export_table(df, stem, index=False):
    csv_path=TAB/f"{stem}.csv"
    tex_path=TAB/f"{stem}.tex"
    df.to_csv(csv_path,index=index)
    try:
        tex_path.write_text(
            df.to_latex(index=index,float_format=lambda x:f"{x:.4g}"),
            encoding="utf-8"
        )
    except Exception as exc:
        print("LaTeX export warning:",exc)
    print("Saved table:",stem)

# Table S0 — reproducible place-name gazetteer used on publication maps.
place_table = PLACE_GAZETTEER[
    ["name","lat","lon","priority","east_km","north_km"]
].copy()
export_table(place_table,"TableS00_Map_place_gazetteer")

# Table S1 — event/source.
table_s1=pd.DataFrame([{
    "Event ID":EVENT_ID,
    "Latitude (°N)":EVENT_LAT,
    "Longitude (°E)":EVENT_LON,
    "Depth (km)":EVENT_DEPTH_KM,
    "Magnitude used":EVENT_MAG,
    "Polarities used":N_POL,
    "Azimuth gap (°)":AZ_GAP,
    "Solution status":SOLUTION_STATUS,
    "NP1 strike":NP1["strike"],
    "NP1 dip":NP1["dip"],
    "NP1 rake":NP1["rake"],
    "NP2 strike":NP2["strike"],
    "NP2 dip":NP2["dip"],
    "NP2 rake":NP2["rake"],
    "Assumed stress drop (MPa)":STRESS_DROP_MPA,
    "Effective friction":EFFECTIVE_FRICTION,
    "Source length (km)":SOURCE_LENGTH_KM,
    "Source width (km)":SOURCE_WIDTH_KM,
    "Mean slip (m)":SOURCE_SLIP_M,
}])
export_table(table_s1,"TableS01_Event_source_and_model_parameters")

# Table S2 — polarity observations.
cols=[
    "network","station","channel","epicentral_km","azimuth_deg",
    "takeoff_angle_deg","polarity","predicted_polarity",
    "correct","snr","confidence"
]
table_s2=polarity[[c for c in cols if c in polarity.columns]].copy()
export_table(table_s2,"TableS02_First_motion_polarities")

# Table S3 — nodal planes.
export_table(nodal,"TableS03_Nodal_planes")

# Table S4 — parent CFS summary.
export_table(cfs_parent,"TableS04_Coulomb_summary_10km")

# 9. Multi-depth Coulomb summary table

In [ ]:
if depth_fields:
    rows=[]
    for dep,(E,N,C) in depth_fields.items():
        rows.append({
            "receiver_depth_km":dep,
            "min_CFS_MPa":float(np.nanmin(C)),
            "max_CFS_MPa":float(np.nanmax(C)),
            "median_CFS_MPa":float(np.nanmedian(C)),
            "p95_abs_CFS_MPa":float(np.nanpercentile(np.abs(C),95)),
            "positive_area_fraction":float(np.mean(C>0)),
            "negative_area_fraction":float(np.mean(C<0)),
        })
    table_depth=pd.DataFrame(rows)
    export_table(table_depth,"TableS05_NP1_Coulomb_depth_summary")
    display(table_depth)

# 10. Mechanism-quality table

This table is deliberately explicit about what is strong and what still requires improvement before final publication.

In [ ]:
quality_table=pd.DataFrame([
    ["Number of polarities",N_POL,"Target ≥8–12 well-distributed observations","Needs improvement" if N_POL<8 else "Acceptable"],
    ["Maximum azimuth gap",f"{AZ_GAP:.1f}°","Prefer <180°, ideally substantially smaller","Needs improvement" if AZ_GAP>=180 else "Acceptable"],
    ["Polarity fit","0/5 parent misfit","Low misfit is desirable","Good fit but non-unique"],
    ["Near-best solutions",len(near),"Fewer/narrower families indicate stronger constraint","Highly non-unique"],
    ["Instrument response","StationXML corrected in parent workflow","Required for consistent waveform polarity/amplitude","Implemented"],
    ["Nodal-plane identity","NP1 and NP2 retained","Independent geological/geodetic constraint required","Unresolved"],
    ["Finite source","Mw/stress-drop scaled rectangle","Finite-fault/geodetic source preferred","Exploratory assumption"],
    ["Coulomb sensitivity","Optional in parent notebook","Stress drop, friction, depth, receiver orientation should be tested","Recommended before paper"],
],columns=["Diagnostic","Current result","Publication expectation","Assessment"])

export_table(quality_table,"TableS06_Quality_control_and_publication_readiness")
display(quality_table)

# Figure S13 — high-resolution table graphic for supplementary PDF

In [ ]:
show=quality_table.copy()

fig,ax=plt.subplots(figsize=(18,6.5))
ax.axis("off")

tbl=ax.table(
    cellText=show.values,
    colLabels=show.columns,
    loc="center",
    cellLoc="left",
    colLoc="center",
    colWidths=[0.18,0.18,0.38,0.22]
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(10.5)
tbl.scale(1,2.2)

for (row,col),cell in tbl.get_celld().items():
    cell.set_edgecolor("0.35")
    cell.set_linewidth(0.8)
    if row==0:
        cell.set_text_props(fontweight="bold",ha="center")
        cell.set_height(cell.get_height()*1.15)

ax.set_title(
    "Mechanism and Coulomb-model quality-control summary",
    fontsize=18,fontweight="bold",pad=20
)

save_figure(fig,"FigS13_Quality_control_table")
plt.show()

# 11. Caption-ready figure index

This creates a concise file that can be pasted into the Supplementary Material section of a manuscript.

In [ ]:
figure_index = pd.DataFrame([
    ["Fig. S1","Regional station geometry and first-motion observations on a real basemap.","Station distribution, acquisition radius, polarities, focal-mechanism inset and explicit regional place labels."],
    ["Fig. S2","Focal-sphere and azimuth coverage.","Shows the observational geometry and the maximum azimuth gap."],
    ["Fig. S3","Focal-mechanism solution density.","Displays non-uniqueness of near-best strike, dip and rake solutions."],
    ["Fig. S4","NP1 Coulomb stress on a real basemap.","Exploratory NP1 ΔCFS map with explicit city/town labels at the parent receiver depth."],
    ["Fig. S5","NP2 Coulomb stress on a real basemap.","Exploratory NP2 ΔCFS map with explicit city/town labels at the parent receiver depth."],
    ["Fig. S6","NP1/NP2 comparison.","Alternative nodal-plane Coulomb scenarios shown on one quantitative scale."],
    ["Fig. S7","Multi-depth ΔCFS slices.","NP1 Coulomb-stress field at 2, 5, 10, 15 and 20 km receiver depths."],
    ["Fig. S8","Static 3-D Coulomb stack.","Depth-stacked NP1 stress surfaces for manuscript visualization."],
    ["Fig. S9","3-D high-amplitude stress lobes.","Strong positive/negative stress-change regions relative to the approximate source plane."],
    ["Fig. S10","Vertical Coulomb cross-sections.","Along-strike and across-strike NP1 vertical sections."],
    ["Fig. S11","Interactive 3-D Coulomb volume.","HTML author/supplement exploration product."],
    ["Fig. S12","Analysis workflow.","Waveform-to-focal-mechanism-to-Coulomb workflow diagram."],
    ["Fig. S13","Quality-control table.","Graphical summary of strengths and present limitations."],
],columns=["Figure","Suggested caption title","Purpose"])

export_table(figure_index,"Supplementary_figure_index")
display(figure_index)

# 12. Output inventory and reproducibility record

In [ ]:
inventory=[]
for p in sorted(PUB.rglob("*")):
    if p.is_file():
        inventory.append({
            "relative_path":str(p.relative_to(PUB)),
            "size_MB":p.stat().st_size/(1024**2)
        })

inventory=pd.DataFrame(inventory)
inventory.to_csv(PUB/"PUBLICATION_OUTPUT_INVENTORY.csv",index=False)

metadata={
    "event_id":EVENT_ID,
    "solution_status":SOLUTION_STATUS,
    "n_polarities":N_POL,
    "azimuth_gap_deg":AZ_GAP,
    "near_best_solution_count":int(len(near)),
    "NP1":NP1,
    "NP2":NP2,
    "stress_drop_MPa":STRESS_DROP_MPA,
    "effective_friction":EFFECTIVE_FRICTION,
    "figure_png_dpi":600,
    "vector_exports":["PDF","SVG"],
    "real_basemap_provider":BASEMAP_NAME,
    "publication_ready_mechanism":bool(publication_ready_mechanism),
    "place_labels":PLACE_GAZETTEER[["name","lat","lon","priority"]].to_dict("records"),
    "place_label_note":"Curated static gazetteer used so labels remain readable above Coulomb overlays; verify before final submission.",
}
(PUB/"REPRODUCIBILITY_METADATA.json").write_text(
    json.dumps(metadata,indent=2),encoding="utf-8"
)

print("Generated files:",len(inventory))
display(inventory)

# Recommended manuscript use

### Figures appropriate now as exploratory/supplementary
- station geometry;
- first-motion coverage/azimuth-gap diagnostic;
- mechanism non-uniqueness;
- NP1/NP2 alternative Coulomb scenarios;
- 3-D model visualization;
- workflow and quality-control diagrams.

### Before treating the mechanism/Coulomb result as a main-paper conclusion
1. Manually review the five current polarities and attempt to recover additional trustworthy stations.
2. Reduce the maximum azimuth gap below 180°, preferably much lower.
3. Compare the polarity mechanism with an independent regional waveform moment tensor.
4. Identify the physical rupture plane using mapped faults, relocated seismicity, InSAR/GNSS or another independent constraint.
5. Replace the assumed rectangular source with a measured finite-source/geodetic solution if available.
6. Run sensitivity to stress drop, effective friction, receiver depth and mapped receiver-fault orientation.
7. Do not interpret a positive ΔCFS lobe as an earthquake prediction.

High-resolution graphics improve presentation, but **scientific constraint and uncertainty treatment are what will matter most to reviewers**.